In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import date

USERNAME = "marclambertsmedia@gmail.com"
PASSWORD = "Meneertosti@1!"
BASE_URL = "https://api.impect.com/v5/customerapi"
AUTH_URL = "https://login.impect.com/auth/realms/production/protocol/openid-connect/token"

def unwrap(r):
    r.raise_for_status()
    return r.json()["data"]

resp = requests.post(AUTH_URL, data={"grant_type":"password","client_id":"api","username":USERNAME,"password":PASSWORD})
resp.raise_for_status()
HEADERS = {"Authorization": f"Bearer {resp.json()['access_token']}"}

# ── Iterations & squads ───────────────────────────────────────────────────────
HK_ITERATION   = 1421   # Czech Fortuna Liga 25/26 (Hradec's league)
REC_ITERATION  = 1454   # Chance Národní Liga 25/26 (recruitment pool)
HK_SQUAD_ID    = 3636   # FC Hradec Kralove

ALL_POSITIONS = ",".join([
    "GOALKEEPER","CENTRAL_DEFENDER","RIGHT_WINGBACK_DEFENDER","LEFT_WINGBACK_DEFENDER",
    "DEFENSE_MIDFIELD","CENTRAL_MIDFIELD","RIGHT_WINGER","LEFT_WINGER",
    "ATTACKING_MIDFIELD","CENTER_FORWARD"
])

# KPI & score name lookups
kpi_names   = {k["id"]: k["name"] for k in unwrap(requests.get(f"{BASE_URL}/kpis",          headers=HEADERS))}
score_names = {s["id"]: s["name"] for s in unwrap(requests.get(f"{BASE_URL}/player-scores",  headers=HEADERS))}

def expand_nested(raw, id_col, val_col, names):
    rows = []
    for rec in raw:
        for item in rec.get(val_col, []):
            rows.append({"playerId": rec["playerId"], "col": names.get(item[id_col], str(item[id_col])), "value": item["value"]})
    return pd.DataFrame(rows).groupby(["playerId","col"])["value"].mean().unstack("col").reset_index() if rows else pd.DataFrame()

def fetch_players(iteration_id, squad_ids):
    """Fetch KPIs + scores for a list of squads, return wide player df."""
    all_kpis, all_scores, all_pos, dur = [], [], [], []
    for squad_id in squad_ids:
        base = f"{BASE_URL}/iterations/{iteration_id}/squads/{squad_id}"
        squad_name = squad_names[iteration_id].get(squad_id, str(squad_id))
        try:
            raw = unwrap(requests.get(f"{base}/player-kpis", headers=HEADERS))
            dur.append(pd.DataFrame([{"playerId": r["playerId"], "squadId": squad_id, "squadName": squad_name,
                "playDuration": r["playDuration"], "matchShare": r["matchShare"], "position": r.get("position")} for r in raw]))
            df = expand_nested(raw, "kpiId", "kpis", kpi_names); df["squadId"] = squad_id; all_kpis.append(df)
        except: pass
        try:
            raw = unwrap(requests.get(f"{base}/player-scores", headers=HEADERS))
            df = expand_nested(raw, "playerScoreId", "playerScores", score_names); df["squadId"] = squad_id; all_scores.append(df)
        except: pass
        try:
            raw = unwrap(requests.get(f"{base}/positions/{ALL_POSITIONS}/player-scores", headers=HEADERS))
            df = expand_nested(raw, "playerScoreId", "playerScores", score_names); df["squadId"] = squad_id; all_pos.append(df)
        except: pass

    dur_df  = pd.concat(dur, ignore_index=True)
    base_df = dur_df.groupby("playerId", as_index=False).agg(
        playDuration=("playDuration","sum"), matchShare=("matchShare","sum"),
        squadId=("squadId","last"), squadName=("squadName","last"), position=("position","last"))

    def avg_wide(frames, suffix=""):
        df = pd.concat(frames, ignore_index=True)
        cols = [c for c in df.columns if c not in ("playerId","squadId")]
        out = df.groupby("playerId", as_index=False)[cols].mean()
        if suffix: out.columns = ["playerId"] + [f"{c}{suffix}" for c in out.columns[1:]]
        return out

    out = base_df
    if all_kpis:   out = out.merge(avg_wide(all_kpis),       on="playerId", how="left")
    if all_scores: out = out.merge(avg_wide(all_scores),      on="playerId", how="left")
    if all_pos:    out = out.merge(avg_wide(all_pos, "_pos"), on="playerId", how="left")
    return out

# Build squad name lookups per iteration
squad_names = {}
for it in [HK_ITERATION, REC_ITERATION]:
    raw = unwrap(requests.get(f"{BASE_URL}/iterations/{it}/squads", headers=HEADERS))
    squad_names[it] = {s["id"]: s["name"] for s in raw}

# Player identity for both iterations
def get_player_info(iteration_id):
    raw = unwrap(requests.get(f"{BASE_URL}/iterations/{iteration_id}/players", headers=HEADERS))
    df  = pd.json_normalize(raw)[["id","firstname","lastname","commonname","birthdate","leg"]]
    df["age"] = pd.to_datetime(df["birthdate"], errors="coerce").apply(
        lambda b: (date.today() - b.date()).days // 365 if pd.notna(b) else None)
    return df.rename(columns={"id":"playerId"})

player_info_hk  = get_player_info(HK_ITERATION)
player_info_rec = get_player_info(REC_ITERATION)

print("✓ Setup complete")
print(f"  Hradec iteration {HK_ITERATION}: {len(squad_names[HK_ITERATION])} squads")
print(f"  Recruitment iteration {REC_ITERATION}: {len(squad_names[REC_ITERATION])} squads")

In [ ]:
# ── PLAYER TRACKING MODEL — FC Hradec Kralove ─────────────────────────────────
# Fetches all squads in HK's league so we can rank HK players vs league peers

all_hk_squads = list(squad_names[HK_ITERATION].keys())
league_df = fetch_players(HK_ITERATION, all_hk_squads)
league_df = league_df.merge(player_info_hk, on="playerId", how="left")

# ── Position-level percentile ranks vs full league ────────────────────────────
KEY_SCORES = [
    "IMPECT_SCORE_PACKING", "IMPECT_SCORE_PXT",
    "OFFENSIVE_IMPECT_SCORE_PACKING", "DEFENSIVE_IMPECT_SCORE_PACKING",
    "PROGRESSION_SCORE_PACKING", "RECEIVING_SCORE_PACKING",
    "INTERVENTIONS_SCORE_PACKING", "SCORER_SCORE",
    "DRIBBLE_SCORE", "LOW_PASS_SCORE", "GROUND_DUEL_SCORE",
]
KEY_SCORES = [c for c in KEY_SCORES if c in league_df.columns]

MIN_MINUTES = 2700   # ~3 full matches minimum

qualified = league_df[league_df["playDuration"] >= MIN_MINUTES].copy()

# Compute percentile rank within position group for each key score
for col in KEY_SCORES:
    qualified[f"{col}_pct"] = qualified.groupby("position")[col].rank(pct=True).round(3)

# Hradec players only
hk_df = qualified[qualified["squadId"] == HK_SQUAD_ID].copy()

front = ["playerId","commonname","position","age","playDuration","matchShare","squadName"]
pct_cols = [f"{c}_pct" for c in KEY_SCORES if f"{c}_pct" in hk_df.columns]
raw_cols = [c for c in KEY_SCORES if c in hk_df.columns]
hk_tracking = hk_df[[c for c in front if c in hk_df.columns] + raw_cols + pct_cols].copy()
hk_tracking = hk_tracking.sort_values("IMPECT_SCORE_PACKING_pct", ascending=False)

hk_tracking.to_excel("hradec_player_tracking.xlsx", index=False)
print(f"✓ Tracking model: {len(hk_tracking)} Hradec players → hradec_player_tracking.xlsx")
display(hk_tracking[["commonname","position","age","IMPECT_SCORE_PACKING","IMPECT_SCORE_PACKING_pct",
                       "OFFENSIVE_IMPECT_SCORE_PACKING_pct","DEFENSIVE_IMPECT_SCORE_PACKING_pct"]].to_string(index=False))

In [ ]:
# ── RECRUITMENT MODEL — Chance Národní Liga (iteration 1454) ──────────────────
# Position-specific key metrics for ranking candidates

POSITION_METRICS = {
    "GOALKEEPER":                  ["IMPECT_SCORE_PACKING","DEFENSIVE_IMPECT_SCORE_PACKING","INTERVENTIONS_SCORE_PACKING"],
    "CENTRAL_DEFENDER":            ["DEFENSIVE_IMPECT_SCORE_PACKING","INTERVENTIONS_SCORE_PACKING","PROGRESSION_SCORE_PACKING","GROUND_DUEL_SCORE"],
    "RIGHT_WINGBACK_DEFENDER":     ["PROGRESSION_SCORE_PACKING","OFFENSIVE_IMPECT_SCORE_PACKING","DEFENSIVE_IMPECT_SCORE_PACKING","LOW_CROSS_SCORE"],
    "LEFT_WINGBACK_DEFENDER":      ["PROGRESSION_SCORE_PACKING","OFFENSIVE_IMPECT_SCORE_PACKING","DEFENSIVE_IMPECT_SCORE_PACKING","LOW_CROSS_SCORE"],
    "DEFENSE_MIDFIELD":            ["IMPECT_SCORE_PACKING","INTERVENTIONS_SCORE_PACKING","LOW_PASS_SCORE","RECEIVING_SCORE_PACKING"],
    "CENTRAL_MIDFIELD":            ["IMPECT_SCORE_PACKING","PROGRESSION_SCORE_PACKING","LOW_PASS_SCORE","RECEIVING_SCORE_PACKING"],
    "RIGHT_WINGER":                ["OFFENSIVE_IMPECT_SCORE_PACKING","PROGRESSION_SCORE_PACKING","DRIBBLE_SCORE","SCORER_SCORE"],
    "LEFT_WINGER":                 ["OFFENSIVE_IMPECT_SCORE_PACKING","PROGRESSION_SCORE_PACKING","DRIBBLE_SCORE","SCORER_SCORE"],
    "ATTACKING_MIDFIELD":          ["OFFENSIVE_IMPECT_SCORE_PACKING","PROGRESSION_SCORE_PACKING","LOW_PASS_SCORE","SCORER_SCORE"],
    "CENTER_FORWARD":              ["OFFENSIVE_IMPECT_SCORE_PACKING","SCORER_SCORE","IMPECT_SCORE_PACKING","RECEIVING_SCORE_PACKING"],
}

all_rec_squads = list(squad_names[REC_ITERATION].keys())
rec_df = fetch_players(REC_ITERATION, all_rec_squads)
rec_df = rec_df.merge(player_info_rec, on="playerId", how="left")

# ── Filters ───────────────────────────────────────────────────────────────────
MIN_MINUTES = 2700   # minimum playing time
MAX_AGE     = 28     # recruitment age ceiling (adjust as needed)

rec_qual = rec_df[
    (rec_df["playDuration"] >= MIN_MINUTES) &
    (rec_df["age"] <= MAX_AGE)
].copy()

# Percentile rank within position for all key scores
all_metrics = list({m for metrics in POSITION_METRICS.values() for m in metrics})
all_metrics = [c for c in all_metrics if c in rec_qual.columns]

for col in all_metrics:
    rec_qual[f"{col}_pct"] = rec_qual.groupby("position")[col].rank(pct=True).round(3)

# Composite recruitment score = mean of position-specific metric percentiles
def recruitment_score(row):
    metrics = POSITION_METRICS.get(row["position"], [])
    pct_vals = [row.get(f"{m}_pct") for m in metrics if pd.notna(row.get(f"{m}_pct"))]
    return round(np.mean(pct_vals), 3) if pct_vals else None

rec_qual["recruitment_score"] = rec_qual.apply(recruitment_score, axis=1)
rec_qual["recruitment_score_pct"] = rec_qual.groupby("position")["recruitment_score"].rank(pct=True).round(3)

# ── Build output: top candidates per position ─────────────────────────────────
front = ["playerId","commonname","age","leg","position","squadName","playDuration","recruitment_score","recruitment_score_pct"]
pct_cols = [f"{c}_pct" for c in all_metrics if f"{c}_pct" in rec_qual.columns]
raw_cols = [c for c in all_metrics if c in rec_qual.columns]
output_cols = [c for c in front if c in rec_qual.columns] + raw_cols + pct_cols

recruitment_df = rec_qual[output_cols].sort_values(["position","recruitment_score_pct"], ascending=[True, False])

recruitment_df.to_excel("hradec_recruitment.xlsx", index=False)
print(f"✓ Recruitment model: {len(recruitment_df)} candidates → hradec_recruitment.xlsx")

# Preview: top 5 per position
print("\nTop 3 per position:")
print(recruitment_df.groupby("position").head(3)[["commonname","age","squadName","position","recruitment_score_pct"]].to_string(index=False))